# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic details and scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}\nVersion: {getattr(metadata, 'version', 'N/A')}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, a RecordSet represents a logical set of records (tables/data frames). Each RecordSet, Field, and Column has a unique `@id`. We'll list all available record sets and their fields.

In [ ]:
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if not record_sets:
    print('No record sets found in the metadata.')
else:
    print('Record Sets and their fields (@id):')
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        rs_obj = dataset.metadata._by_id(rs['@id'])   # Access record set by @id
        field_ids = [f['@id'] for f in rs_obj.fields] if hasattr(rs_obj, 'fields') else []
        print(f"  Fields: {field_ids}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing entities by their `@id`s.

We'll extract all record sets found above. If your dataset shows multiple record sets, you can select one or more to load.

In [ ]:
# Extract data from each record set (by @id)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}.")

# For demonstration, pick the first record set if available
if record_set_ids:
    active_rs_id = record_set_ids[0]
else:
    active_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we'll select a numeric field (e.g., GAD-7 or PHQ-9 score) and group by a demographic field (e.g., gender or village), referencing everything by their `@id`s.

In [ ]:
import numpy as np

# Identify numeric and group fields from columns (using @id)
numeric_field_candidates = ['gad7_score', 'phq9_score', 'psq_score']
group_field_candidates = ['gender', 'village', 'level_of_education']

if active_rs_id:
    df = dataframes[active_rs_id]
    # Pick first available numeric field
    numeric_field = None
    for nf in numeric_field_candidates:
        if nf in df.columns:
            numeric_field = nf
            break
    # Pick first available group field
    group_field = None
    for gf in group_field_candidates:
        if gf in df.columns:
            group_field = gf
            break

    if numeric_field:
        threshold = df[numeric_field].quantile(0.9) if np.issubdtype(df[numeric_field].dtype, np.number) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by demographic field
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data ({numeric_field} mean) by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric scores and visualize group means by demographic attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if active_rs_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Using `mlcroissant`, we've loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, referenced all entities by their `@id`, and performed basic EDA. Survey scores such as GAD-7 and PHQ-9 can be filtered and grouped by demographic attributes for deeper mental health analysis.

This notebook showcased the usage of `mlcroissant` and outlined a reproducible workflow for FAIR AI-ready data packages.